# ⛰️ The Ramachandran Landscape Explorer

The **Ramachandran Plot** is one of the most famous—and initially confusing—concepts in structural biology. 
It is a 2D scatter plot showing the statistically allowed torsion angles for the protein backbone. But why are certain angles forbidden?

In this interactive lab, we pull back the curtain. You will directly manipulate the $\phi$ and $\psi$ torsion angles of a dipeptide. 
We've paired a live **2D Ramachandran topographical map** with a **3D space-filling molecular viewer** and a real-time **Physics Engine**. 

When you drag the angles into "forbidden" territory, you won't just see a dot on a plot move—you'll see the 3D atoms physically crash into each other, and you'll watch the potential energy spike to absurd levels!


In [ ]:
import os
import io
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# synth-pdb internals
from synth_pdb.generator import generate_pdb_content
from synth_pdb.physics import EnergyMinimizer

# Disable overly verbose OpenMM warnings for clean UI
import logging

logging.getLogger("synth_pdb.physics").setLevel(logging.ERROR)

In [ ]:
def render_ramachandran_explorer(phi, psi):
    # 1. GENERATE THE 3D MOLECULE
    # We use a tripeptide (AAA) so the central Alanine has complete phi/psi degrees of freedom.
    # cap_termini=True ensures the ends are properly chemically bonded (ACE/NME).
    pdb_string = generate_pdb_content(
        sequence_str="AAA", phi_list=[phi, phi, phi], psi_list=[psi, psi, psi], cap_termini=True
    )

    # 2. CALCULATE POTENTIAL ENERGY (Detecting steric clashes)
    minimizer = EnergyMinimizer()
    try:
        # Calculate the potential energy without minimizing it
        energy = minimizer.calculate_energy(pdb_string)
    except Exception:
        energy = 99999.9  # Fallback for extreme geometries

    # 3. SETUP THE 2D MATPLOTLIB PLOT
    plt.ioff()  # Prevent auto-display
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)

    # Draw standard allowed regions (approximate boxes for visualization)
    # Alpha Helix (right-handed)
    ax.add_patch(
        Rectangle((-120, -70), 80, 50, facecolor="#3b82f6", alpha=0.4, label="Alpha Helix")
    )
    # Beta Sheet
    ax.add_patch(Rectangle((-150, 90), 90, 80, facecolor="#10b981", alpha=0.4, label="Beta Sheet"))
    # Left-handed helix
    ax.add_patch(Rectangle((40, 40), 60, 50, facecolor="#ef4444", alpha=0.4, label="L-Helix"))

    # Draw the current position
    ax.scatter([phi], [psi], color="black", s=150, zorder=5, edgecolors="white", linewidth=2)

    # Styling
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.set_yticks([-180, -90, 0, 90, 180])
    ax.set_xlabel(r"Phi ($\phi$) Degrees", fontweight="bold")
    ax.set_ylabel(r"Psi ($\psi$) Degrees", fontweight="bold")
    ax.set_title("Ramachandran Map", fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.5)

    # Save plot to buffer
    buf = io.BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    img_widget = widgets.Image(value=buf.read(), format="png")

    # 4. SETUP THE 3D VIEWER
    view = py3Dmol.view(width=400, height=400)
    view.addModel(pdb_string, "pdb")

    # Render with Spacefill to clearly show steric clashes (atoms overlapping)
    view.setStyle({"sphere": {"radius": 1.2}})
    # Highlight the backbone in stick
    view.setStyle({"stick": {"radius": 0.2}}, add=True)

    view.zoomTo()
    view.setBackgroundColor("#1e1e1e")

    # 5. ASSEMBLE UI
    # Energy formatting
    color = "#10b981"  # Green
    status = "RELAXED (Allowed Region)"
    if energy is None:
        energy = 99999.9

    if energy > 500:
        color = "#ef4444"  # Red
        status = "SEVERE CLASH! (Forbidden Region)"
    elif energy > 50:
        color = "#f59e0b"  # Orange
        status = "STRAINED"

    energy_html = widgets.HTML(f"""
    <div style='text-align: center; margin-bottom: 15px; font-family: monospace; background: #2d2d2d; padding: 15px; border-radius: 8px;'>
        <h3 style='margin: 0; color: white;'>Potential Energy: <span style='color: {color};'>{energy:.1f} kJ/mol</span></h3>
        <p style='margin: 5px 0 0 0; color: {color}; font-weight: bold; font-size: 1.2em;'>{status}</p>
    </div>
    """)

    # Use an Output widget to capture py3Dmol's display
    viewer_out = widgets.Output()
    with viewer_out:
        view.show()

    ui = widgets.VBox([energy_html, widgets.HBox([img_widget, viewer_out])])

    display(ui)


print("Visualization engine loaded.")

### 🎛️ The Interactive Forge

Drag the $\phi$ and $\psi$ sliders below. 

**Try this:**
1. Leave the settings at the **Alpha Helix** region ($\phi pprox -60$, $\psi pprox -40$). The energy is low and the atoms fit together perfectly.
2. Drag both sliders to exactly `0°`. Notice how the oxygen and carbon atoms physically overlap (a steric clash), and the physics engine reports a massive spike in potential energy! This is *why* the center of the Ramachandran plot is empty.


In [ ]:
# Run the interactive widget!
# In CI/CD headless environments, this just prints the widget placeholder.
# We set continuous_update=False so the physics engine only runs when you release the mouse button.
widgets.interact(
    render_ramachandran_explorer,
    phi=widgets.IntSlider(
        min=-180, max=180, step=10, value=-60, description=r"Phi ($\phi$):", continuous_update=False
    ),
    psi=widgets.IntSlider(
        min=-180, max=180, step=10, value=-40, description=r"Psi ($\psi$):", continuous_update=False
    ),
);